# Closed-loop stimulation mediation

Analysis source for Figure 5e–g: nested logistic regression, network mediation, regional theta-power mediation, and theta-adjusted network mediation.

See [README](README.md) for inputs and execution notes. Data and saved outputs are not distributed. The calculation code is retained; headings and inaccurate comments have been clarified.

## Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt 
import sklearn.linear_model as lm 
import re
import statsmodels.api as sm 
import pandas as pd
import h5py

## Load closed-loop scores and regional power

In [ ]:
data = h5py.File('ClosedRandomLoop_behavior&scores.mat','r')

In [ ]:
myDict = {}
for key in data.keys():
    # check if key includes 'CL'
    if re.search('CL', key):
        print(key)
        try:
            myDict[key] = data[key].value
        except:
            myDict[key] = data[key]
    # myDict[key] = data[key].value

In [ ]:
alltime_list = [] 
laser_list = [] 
lasertime_list = [] 
norm_list = [] 
t_list = []
t_soft_list = []
t_soft_norm_list = []
t_norm_list = [] 
tuse_list = [] 
usef_list = [] 
uselaser_list = [] 
usetimes_list = []
power=[]
areas=[]
for key in myDict.keys():
    if bool(re.search('_poweruse',key)):
        print('poweruse',key)
        power.append(np.squeeze(myDict[key]))
    if bool(re.search('_AreaNames',key)):
        print('AreaNames',key)
        areas.append(np.squeeze(myDict[key]))
    if bool(re.search('_alltime',key)):
        print('Alltime',key)
        alltime_list.append(np.squeeze(myDict[key])) 
    elif bool(re.search('_lasertime',key)):
        print('Lasertime',key)
        lasertime_list.append(np.squeeze(myDict[key])) 
    # elif bool(re.search('_laser',key)):
    #     print('Laser',key)
    #     laser_list.append(np.squeeze(myDict[key]))
    elif bool(re.search('_t_norm',key)):
        print(myDict['Mouse9332_030421_tuse'].shape)
        print('T_norm',key)
        t_norm_list.append(np.squeeze(myDict[key])) 
    elif bool(re.search('_tuse',key)):
        print('Tuse',key)
        tuse_list.append(np.squeeze(myDict[key])) 
    elif bool(re.search('_all_soft_norm',key)):
        print('T_soft_norm',key)
        t_soft_norm_list.append(np.squeeze(myDict[key])) 
    elif bool(re.search('_all_soft',key)):
        print('T_soft',key)
        t_soft_list.append(np.squeeze(myDict[key])) 
    elif bool(re.search('_t',key)):
        print('T',key)
        t_list.append(np.squeeze(myDict[key])) 
    elif bool(re.search('_usef',key)):
        print('usef',key)
        usef_list.append(np.squeeze(myDict[key].T)) 
    elif bool(re.search('_all_laser',key)):
        print('all_laser',key)
        uselaser_list.append(np.squeeze(myDict[key])) 
    # elif bool(re.search('_uselaser',key)):
    #     print('uselaser',key)
    #     uselaser_list.append(np.squeeze(myDict[key])) 
    elif bool(re.search('_usetimes',key)):
        print('usetimes',key)
        usetimes_list.append(np.squeeze(myDict[key])) 
    else:
        print('!!!!!!!!!!!!!!!!!!!!!!!!')
        print('Unmatched',key)

usef_list=alltime_list

In [ ]:
data = h5py.File('Closed_loop_singleregionMediation_fixed.mat','r')
myDict = {}
for key in data.keys():
    # check if key includes 'CL'
    if re.search('CL', key):
        print(key)
        try:
            myDict[key] = data[key].value
        except:
            myDict[key] = data[key]
    # myDict[key] = data[key].value



In [ ]:
power=[]
all_labels=[]
for key in myDict.keys():
    print('current key',key)
    if bool(re.search('_poweruse',key)):
        print('poweruse',key)
        power.append(np.squeeze(myDict[key]))
    elif bool(re.search('_all_labels',key)):
        print('all_labels',key)
        all_labels.append(np.squeeze(myDict[key]))
    else:
        print('!!!!!!!!!!!!!!!!!!!!!!!!')
        print('Unmatched',key)

In [ ]:
uselaser_list[1].shape

## Aggregate theta-frequency bins

In [ ]:
# Sum the frequency-bin slice specified below across all available brain regions.
lowfreq=4
highfreq=11
power_processed=power.copy()
for i in range(len(power)):
    n=len(uselaser_list[i])
    n2=power[i].shape[0]
    if n2<n:
        print('!!!!!!!!!!!!!!!!!!!!!!!!')
        print('n2<n',i,n2,n)  
    if n<n2:
        n=n2
        uselaser_list[i]=uselaser_list[i][0:n]

    #     n=n2
    #     uselaser_list[i]=uselaser_list[i][0:n]
    power_processed[i]=np.sum(power[i][0:n,:,lowfreq:highfreq+1],axis=2).T

# concatenate power processed
power_processed=np.concatenate(power_processed,axis=1)
    


In [ ]:
power_processed.shape

In [ ]:
for i in range(len(uselaser_list)):
    n=len(uselaser_list[i])
    usef_list[i] = usef_list[i][0:n,:]

for i,t_soft_entry in enumerate(t_soft_list):
    t_soft_list[i] = t_soft_entry[0:usef_list[i].shape[0]]

In [ ]:
# define a numerically stable softplus in numpy
def softplus(x):
    sx=np.zeros(x.shape)
    for i in range(len(x)):
        if x[i]>15:
            sx[i]=x[i]
        elif x[i]<=15:
            sx[i]=np.log(1+np.exp(x[i]))
    return sx


In [ ]:
# ndx=np.zeros(len(usef_list),dtype=bool)
# for i, usef in enumerate(usef_list):
#     # print(usef.shape)
#     if usef.shape[1]==3:
#         ndx[i]=True
# ndx=np.where(ndx)[0]
# uselaser_list=np.array(uselaser_list)[ndx]
# usef_list=np.array(usef_list)[ndx]
# usetimes_list=np.array(usetimes_list)[ndx]
# # keep only the first column of each entry in t_soft_list

# # tuse_list=np.array(tuse_list)
# # t_soft_list=np.array(t_soft_list)

In [ ]:
power_processed.shape

## Select blue/yellow stimulated windows

In [ ]:
uselaser = np.concatenate(uselaser_list) 
usef = np.hstack(usef_list).T
tuse = np.concatenate(t_soft_norm_list)
condition = usef[:,1]
behavior = usef[:,2]
# Blue and yellow stimulated windows in the intact-male condition.
idx_select = ((uselaser==1)|(uselaser==2))&(condition==4) 

# all lasers
# idx_select = ((uselaser==0)|(uselaser==1)|(uselaser==2))&(condition==4)

# behavior_select = (behavior==1)|(behavior==2)
behavior_select = (behavior==1)|(behavior==2)|(behavior==0)
idx_total = idx_select & behavior_select
idx_total_num = np.where(idx_total)[0]
idx_total_shift=idx_total_num-1
idx_total = np.where(idx_total)[0]
# network_sub = tuse[idx_total]
network_sub = tuse[idx_total]
power_sub=power_processed[:,idx_total_num]
network_sub_1s_before = tuse[idx_total_shift]
stim_sub = uselaser[idx_total] 
# convert entries of 2 to 0 in stim_sub
stim_sub[stim_sub==2] = 0
behavior_sub = behavior[idx_total] 
behavior_sub[behavior_sub==2] = 0
# mouse_sub = mouse_id[idx_total]

In [ ]:
plt.hist(network_sub, bins=100)
plt.show()

In [ ]:
# make histogram of behavior_sub for blue and yellow lasers
plt.figure()
plt.hist([behavior_sub[stim_sub==0],behavior_sub[stim_sub==1]],bins=2,color=['yellow','blue'],density=True)

## Nested logistic regression — Figure 5e

In [ ]:
# mouse_dummy=pd.get_dummies(mouse_sub)
X1 = pd.DataFrame(data={'Stimulation':stim_sub})
X2 = pd.DataFrame(data={'Stimulation':stim_sub,'Network':network_sub})

# X1 = pd.DataFrame(data={'Stimulation':stim_sub,'Network_before':network_sub_1s_before})
# X2 = pd.DataFrame(data={'Stimulation':stim_sub,'Network_before':network_sub_1s_before,'Network':network_sub}) 
Y = pd.DataFrame(data={'Behavior':behavior_sub})

# check for nans in x2, and remove from x1,x2, and y
# print(np.where(np.isnan(X2)))
contains_nans=np.where(np.isnan(X2))
X1=X1.drop(contains_nans[0])
X2=X2.drop(contains_nans[0])
Y=Y.drop(contains_nans[0])


In [ ]:
X1_2 = sm.add_constant(X1)
# X1_3 = pd.merge(X1,mouse_dummy,how='right',left_index=True,right_index=True)
reduced_model = sm.Logit(Y,X1_2).fit()

In [ ]:
reduced_ll = reduced_model.llf 
print('Log likelihood',reduced_ll)

In [ ]:
reduced_model.summary()

In [ ]:
X2_2 = sm.add_constant(X2) 
# X2_3 = pd.merge(X2,mouse_dummy,how='right',left_index=True,right_index=True)
full_model = sm.Logit(Y,X2_2).fit() 
full_ll = full_model.llf

In [ ]:
print(full_ll)

In [ ]:
full_model.summary()

In [ ]:
LR_statistic = -2*(reduced_ll-full_ll) 
print('Likelihood_ratio stat',LR_statistic) 
from scipy.stats import chi2
p_val = chi2.sf(LR_statistic,1) 
print('P-value',p_val)

In [ ]:
print(Y.shape)
print(network_sub.shape)
print(stim_sub.shape)

## Network causal mediation — Figure 5e

In [ ]:
## causal mediation analysis

# import links from statsmodels
import statsmodels.genmod.families.links as links
df = pd.DataFrame({'behavior':behavior_sub,'network':network_sub,'stimulation':stim_sub})
# drop nans from df
df=df.dropna()
# Use the supplied network scores without an additional transformation.
# df['network_norm'] = np.sqrt(df['network'])
# df['network_norm'] = quantile_transform(df['network'].values.reshape(-1,1),n_quantiles=25,output_distribution='uniform',copy=True).reshape(-1)
df['network_norm']=df['network']
# add constant to df
df['const']=1

# # use quantile normalization on network scores in df
# from sklearn.preprocessing import quantile_transform
# df['network_norm'] = quantile_transform(df['network'].values.reshape(-1,1),n_quantiles=1000,output_distribution='normal',copy=True).reshape(-1)

In [ ]:
probit = links.probit
outcome_model = sm.GLM.from_formula("behavior ~ network_norm + stimulation + const",
                                    df, family=sm.families.Binomial(link=probit()))
mediator_model = sm.OLS.from_formula("network_norm ~ stimulation + const", df)
causal_model= sm.stats.Mediation(outcome_model, mediator_model, exposure="stimulation",mediator="network_norm").fit()
# pull out ACME and ADE
acme = np.mean(causal_model.ACME_avg)
ade = np.mean(causal_model.ADE_avg)

In [ ]:
causal_model.summary()

## Regional theta-power mediation — Figure 5f

In [ ]:
for br in range(power_processed.shape[0]):
    df = pd.DataFrame({'behavior':behavior_sub,'network':power_sub[br],'stimulation':stim_sub})
    # drop nans from df
    df=df.dropna()
    # Use the supplied network scores without an additional transformation.
    df['network_norm']=df['network']
    # add constant to df
    df['const']=1
    probit = links.probit
    outcome_model = sm.GLM.from_formula("behavior ~ network_norm + stimulation + const",
                                        df, family=sm.families.Binomial(link=probit()))
    mediator_model = sm.OLS.from_formula("network_norm ~ stimulation + const", df)
    causal_model= sm.stats.Mediation(outcome_model, mediator_model, exposure="stimulation",mediator="network_norm").fit()
    # pull out ACME and ADE
    acme = np.mean(causal_model.ACME_avg)
    ade = np.mean(causal_model.ADE_avg)
    print('in brain region',br)
    print(causal_model.summary())

## Network mediation adjusted for regional theta power — Figure 5g

In [ ]:
for br in range(power_processed.shape[0]):
    df = pd.DataFrame({'behavior':behavior_sub,'network':network_sub,'intermediate':power_sub[br],'stimulation':stim_sub})
    # drop nans from df
    df=df.dropna()
    # Use the supplied network scores without an additional transformation.
    df['network_norm']=df['network']
    # add constant to df
    df['const']=1
    probit = links.probit
    outcome_model = sm.GLM.from_formula("behavior ~ network_norm + intermediate + stimulation + const",
                                        df, family=sm.families.Binomial(link=probit()))
    mediator_model = sm.OLS.from_formula("network_norm ~ stimulation + intermediate + const", df)
    causal_model= sm.stats.Mediation(outcome_model, mediator_model, exposure="stimulation",mediator="network_norm").fit()
    # pull out ACME and ADE
    acme = np.mean(causal_model.ACME_avg)
    ade = np.mean(causal_model.ADE_avg)
    print('in brain region',br)
    print(causal_model.summary())

## Region labels

In [ ]:
areas=['IL',
'LHb',
'LSN',
'MDThal',
'MeA',
'NAc',
'OFC',
'PL',
'V1',
'VHipp',
'VMHvl']

In [ ]:
areas[6]

## Additional exploratory model

This model adjusts for all eleven regions together; it is not the per-region model shown in Figure 5g.

In [ ]:
df = pd.DataFrame({'behavior':behavior_sub,'network':network_sub,'intermediate0':power_sub[0],'intermediate1':power_sub[1],
                   'intermediate2':power_sub[2],'intermediate3':power_sub[3],'intermediate4':power_sub[4],'intermediate5':power_sub[5],
                     'intermediate6':power_sub[6],'intermediate7':power_sub[7],'intermediate8':power_sub[8],'intermediate9':power_sub[9],
                          'intermediate10':power_sub[10],               
                   'stimulation':stim_sub})
# drop nans from df
df=df.dropna()
# Use the supplied network scores without an additional transformation.
df['network_norm']=df['network']
# add constant to df
df['const']=1
probit = links.probit
outcome_model = sm.GLM.from_formula("behavior ~ network_norm + intermediate0 + intermediate1 + intermediate2 + intermediate3 + intermediate4 + intermediate5 + intermediate6 + intermediate7 + intermediate8 + intermediate9 + intermediate10 + stimulation + const",
                                    df, family=sm.families.Binomial(link=probit()))
mediator_model = sm.OLS.from_formula("network_norm ~ stimulation + intermediate0                                      + intermediate1 + intermediate2 + intermediate3 + intermediate4 + intermediate5 + intermediate6 + intermediate7 + intermediate8 + intermediate9 + intermediate10                                      + const", df)
causal_model= sm.stats.Mediation(outcome_model, mediator_model, exposure="stimulation",mediator="network_norm").fit()
# pull out ACME and ADE
acme = np.mean(causal_model.ACME_avg)
ade = np.mean(causal_model.ADE_avg)
print('in brain region',br)
print(causal_model.summary())